# 🐴 バックテスト

学習済みモデルの回収率を過去データで検証します。

In [ ]:
!pip install supabase lightgbm scikit-learn matplotlib -q

In [ ]:
SUPABASE_URL = 'https://infypumigexmpdmijhnx.supabase.co'
SUPABASE_KEY = 'eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9...'  # service_role key
from supabase import create_client
import pandas as pd, numpy as np, joblib, matplotlib.pyplot as plt
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print('✅ 接続OK')

In [ ]:
model = joblib.load('keiba_model_XXXXXXXX.pkl')  # ファイル名を変更
print('✅ モデル読み込みOK')

In [ ]:
all_data = []
offset = 0
while True:
    res = supabase.table('v_features').select('*').gte('race_date', '2026-01-01').range(offset, offset+999).execute()
    if not res.data: break
    all_data.extend(res.data)
    offset += 1000
df = pd.DataFrame(all_data)
df['race_date'] = pd.to_datetime(df['race_date'])
print(f'✅ テストデータ: {len(df):,}件')

In [ ]:
from sklearn.preprocessing import LabelEncoder
cat_cols = ['venue_code', 'surface', 'track_condition', 'weather', 'class']
for col in cat_cols:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))
feature_cols = [
    'venue_code_enc', 'distance', 'surface_enc', 'track_condition_enc',
    'weather_enc', 'post_position', 'entry_count', 'class_enc',
    'avg_finish_3', 'avg_finish_5', 'fukusho_rate_3', 'fukusho_rate_5',
    'last_finish', 'rest_days', 'distance_diff', 'horse_weight', 'horse_weight_diff',
    'jockey_fukusho_rate', 'trainer_fukusho_rate',
    'father_fukusho_rate', 'mother_father_fukusho_rate',
    'odds', 'popularity', 'popularity_ratio',
]
df['proba'] = model.predict_proba(df[feature_cols])[:, 1]

In [ ]:
res = supabase.table('payouts').select('race_id,combination,payout').eq('bet_type', '複勝').execute()
df_pay = pd.DataFrame(res.data)
df_pay['horse_number'] = df_pay['combination'].str.extract(r'(\d+)').astype(float)
df_pay['fukusho_odds'] = df_pay['payout'] / 100
df = df.merge(df_pay[['race_id','horse_number','fukusho_odds']], on=['race_id','horse_number'], how='left')
df['expected_value'] = df['proba'] * df['fukusho_odds'].fillna(0)
df['hit'] = (df['target'] == 1).astype(int)
print('✅ 期待値算出完了')

In [ ]:
print('=' * 50)
print(f"{'閾値':>8} | {'購入件数':>8} | {'的中率':>8} | {'回収率':>8}")
print('-' * 50)
for threshold in [0.8, 1.0, 1.1, 1.2, 1.3, 1.5]:
    buy = df[df['expected_value'] > threshold].copy()
    if len(buy) == 0: continue
    total_bet    = len(buy) * 100
    total_return = (buy[buy['hit']==1]['fukusho_odds'] * 100).sum()
    recovery     = total_return / total_bet * 100
    hit_rate     = buy['hit'].mean() * 100
    print(f'{threshold:>8.1f} | {len(buy):>8,} | {hit_rate:>7.1f}% | {recovery:>7.1f}%')
print('=' * 50)

In [ ]:
buy = df[df['expected_value'] > 1.0].copy()
buy['month'] = buy['race_date'].dt.to_period('M')
monthly = buy.groupby('month').apply(
    lambda x: (x[x['hit']==1]['fukusho_odds']*100).sum() / (len(x)*100) * 100 if len(x) > 0 else 0
)
plt.figure(figsize=(12, 5))
monthly.plot(kind='bar', color=['green' if v >= 100 else 'red' for v in monthly])
plt.axhline(y=100, color='black', linestyle='--', label='回収率100%')
plt.title('月別回収率推移（期待値 > 1.0）', fontsize=14)
plt.ylabel('回収率 (%)')
plt.legend()
plt.tight_layout()
plt.show()
print(f'\n平均回収率: {monthly.mean():.1f}%')